In [1]:
import pprint
import radiate as rd
import numpy as np
import polars as pl

rd.random.seed(123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

x = -1.0
for _ in range(20):
    x += 0.1
    inputs.append([x])
    answers.append([compute(x)])

X = np.array(inputs, dtype=np.float32)  # (N, 1)
Y = np.array(answers, dtype=np.float32)  # (N, 1)

# Add bias term: (N, 2) = [x, 1]
Xb = np.concatenate([X, np.ones((X.shape[0], 1), dtype=np.float32)], axis=1)


def fit(weights: list[np.ndarray]) -> float:
    # Decode weights
    W1 = weights[0].reshape((8, 2))
    W2 = weights[1].reshape((8, 8))
    W3 = weights[2].reshape((1, 8))

    # Forward pass
    # Xb: (N,2)
    h1 = Xb @ W1.T  # (N,2) @ (2,8) => (N,8)
    h1 = np.maximum(0, h1)  # ReLU activation

    h2 = h1 @ W2  # (N,8) @ (8,8) => (N,8)
    h2 = np.tanh(h2)  # tanh activation

    yhat = h2 @ W3.T  # (N,8) @ (8,1) => (N,1)

    # MSE
    return float(np.mean((yhat - Y) ** 2, dtype=np.float32))

In [3]:
collector = rd.MetricCollector()
engine = (
    rd.Engine.float(
        shape=[16, 64, 8],
        init_range=(-1.0, 1.0),
        bounds=(-3.0, 3.0),
        use_numpy=True,
        dtype=rd.Float32,
    )
    .fitness(fit)
    .subscribe(collector)
    .minimizing()
    .select(rd.Select.boltzmann(temp=4.0))
    .alters(rd.Cross.blend(0.7, 0.4), rd.Mutate.gaussian(0.1))
    .limit(rd.Limit.score(0.01), rd.Limit.generations(500))
)

result = engine.run(log=True)

2026-04-09T02:32:31.544859Z  INFO Epoch 1    | Score:   0.8838 | Time: 2.22ms
2026-04-09T02:32:31.545814Z  INFO Epoch 2    | Score:   0.8670 | Time: 3.12ms
2026-04-09T02:32:31.546728Z  INFO Epoch 3    | Score:   0.8588 | Time: 3.98ms
2026-04-09T02:32:31.547781Z  INFO Epoch 4    | Score:   0.6303 | Time: 4.97ms
2026-04-09T02:32:31.548670Z  INFO Epoch 5    | Score:   0.6078 | Time: 5.81ms
2026-04-09T02:32:31.549527Z  INFO Epoch 6    | Score:   0.4879 | Time: 6.63ms
2026-04-09T02:32:31.550390Z  INFO Epoch 7    | Score:   0.4633 | Time: 7.46ms
2026-04-09T02:32:31.551240Z  INFO Epoch 8    | Score:   0.4245 | Time: 8.27ms
2026-04-09T02:32:31.552067Z  INFO Epoch 9    | Score:   0.3829 | Time: 9.06ms
2026-04-09T02:32:31.553087Z  INFO Epoch 10   | Score:   0.2610 | Time: 10.03ms
2026-04-09T02:32:31.554008Z  INFO Epoch 11   | Score:   0.1998 | Time: 10.89ms
2026-04-09T02:32:31.554887Z  INFO Epoch 12   | Score:   0.1998 | Time: 11.74ms
2026-04-09T02:32:31.555760Z  INFO Epoch 13   | Score:   0.199

In [4]:
metrics = result.metrics()
df = metrics.to_polars()
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,version,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""diversity_ratio""",0.92,63.59,0.949104,0.024663,0.000608,0.0,0.88,0.99,67,null,null,null,null,null,null,67,1,"[""derived"", ""statistic""]"
"""lineage_events""",80.0,5360.0,80.0,0.0,0.0,0.0,80.0,80.0,67,null,null,null,null,null,null,67,1,"[""age"", ""statistic"", ""lineage""]"
"""alter_within_family""",133.0,8931.0,51.034286,62.908062,3957.424316,0.533711,1.0,142.0,175,null,null,null,null,null,null,67,1,"[""alterer"", ""statistic"", … ""lineage""]"
"""unique_scores""",91.0,6340.0,94.626862,2.569258,6.601086,-0.190965,88.0,99.0,67,null,null,null,null,null,null,67,1,"[""derived"", ""statistic"", ""score""]"
"""unique_members""",92.0,6359.0,94.910446,2.466327,6.082767,-0.235378,88.0,99.0,67,null,null,null,null,null,null,67,1,"[""derived"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""best_score_improvement""",1.0,33.0,1.0,0.0,0.0,0.0,1.0,1.0,33,null,null,null,null,null,null,67,1,"[""statistic"", ""score""]"
"""audit_step""",0.0,null,null,null,null,null,null,null,67,1320µs,19µs,9µs,13µs,60µs,0µs,67,1,"[""time"", ""step""]"
"""blend_crossover""",1004.0,75484.0,1126.626831,127.470116,16248.630859,0.175426,868.0,1481.0,67,1179µs,17µs,4µs,13µs,44µs,0µs,67,2,"[""alterer"", ""crossover"", … ""time""]"


In [5]:
print(result.metrics().dashboard())
# pprint.pprint(metrics["carryover_rate"].tags())

for metric in metrics.values_by_tag(rd.Tag.DERIVED):
    print(metric)

print()
pprint.pprint(metrics["carryover_rate"].to_dict())

[31 metrics, 2044 updates]
----- Metrics -----
  carryover: 0.154  diversity: 0.949  unique_members: 92  unique_scores: 91  improvements: 33  iter_time(mean): 857.445µs

== Statistics ==
Name                     | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt       | Entr      
-------------------------------------------------------------------------------------------------------------------------------------------------
age                      | value  | 0.400      | 0.000      | 12.000     | 6700   | -            | 1.152      | 39.478     | 539.112    | -         
alter_cross_family       | value  | 25.148     | 1.000      | 62.000     | 142    | -            | 25.533     | 0.206      | 1.147      | -         
alter_parent_reuse       | value  | 2.703      | 1.000      | 60.000     | 3304   | -            | 5.248      | 5.677      | 40.642     | -         
alter_within_family      | value  | 51.034     | 1.000      | 142.000  

In [6]:
import ipywidgets as widgets
from IPython.display import display
from collections import deque


class NotebookRunView:
    def __init__(self, max_log_lines=200):
        self.status = widgets.HTML("<b>Status:</b> Starting...")
        self.progress = widgets.IntProgress(
            value=0, min=0, max=100, description="Epoch"
        )
        self.summary = widgets.HTML("")
        self.log = widgets.HTML(
            value="<pre style='margin:0'></pre>",
            layout=widgets.Layout(
                border="1px solid #ddd",
                height="240px",
                overflow_y="auto",
                width="100%",
            ),
        )
        self.lines = deque(maxlen=max_log_lines)

        self.root = widgets.VBox(
            [
                self.status,
                self.progress,
                self.summary,
                self.log,
            ]
        )

    def display(self):
        display(self.root)

    def update(self, epoch, total_epochs, score, elapsed_ms):
        self.progress.max = total_epochs
        self.progress.value = epoch
        self.status.value = "<b>Status:</b> Running"
        self.summary.value = (
            f"<b>Epoch:</b> {epoch} / {total_epochs} "
            f"&nbsp;&nbsp; <b>Score:</b> {score:.4f} "
            f"&nbsp;&nbsp; <b>Time:</b> {elapsed_ms:.2f} ms"
        )

        self.lines.append(
            f"Epoch {epoch:>4} | Score: {score:>8.4f} | Time: {elapsed_ms:>8.2f} ms"
        )
        joined = "\n".join(self.lines)
        self.log.value = f"<pre style='margin:0;padding:0'>{joined}</pre>"

    def finish(self):
        self.status.value = "<b>Status:</b> Done"

In [7]:
view = NotebookRunView()
view.display()

for epoch in range(1, 101):
    score = 1 / epoch
    elapsed_ms = epoch * 2.3
    view.update(epoch, 100, score, elapsed_ms)

view.finish()
view.status

HTML(value='<b>Status:</b> Done')

In [8]:
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,version,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""diversity_ratio""",0.92,63.59,0.949104,0.024663,0.000608,0.0,0.88,0.99,67,null,null,null,null,null,null,67,1,"[""derived"", ""statistic""]"
"""lineage_events""",80.0,5360.0,80.0,0.0,0.0,0.0,80.0,80.0,67,null,null,null,null,null,null,67,1,"[""age"", ""statistic"", ""lineage""]"
"""alter_within_family""",133.0,8931.0,51.034286,62.908062,3957.424316,0.533711,1.0,142.0,175,null,null,null,null,null,null,67,1,"[""alterer"", ""statistic"", … ""lineage""]"
"""unique_scores""",91.0,6340.0,94.626862,2.569258,6.601086,-0.190965,88.0,99.0,67,null,null,null,null,null,null,67,1,"[""derived"", ""statistic"", ""score""]"
"""unique_members""",92.0,6359.0,94.910446,2.466327,6.082767,-0.235378,88.0,99.0,67,null,null,null,null,null,null,67,1,"[""derived"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""best_score_improvement""",1.0,33.0,1.0,0.0,0.0,0.0,1.0,1.0,33,null,null,null,null,null,null,67,1,"[""statistic"", ""score""]"
"""audit_step""",0.0,null,null,null,null,null,null,null,67,1320µs,19µs,9µs,13µs,60µs,0µs,67,1,"[""time"", ""step""]"
"""blend_crossover""",1004.0,75484.0,1126.626831,127.470116,16248.630859,0.175426,868.0,1481.0,67,1179µs,17µs,4µs,13µs,44µs,0µs,67,2,"[""alterer"", ""crossover"", … ""time""]"


In [ ]:
import polars.selectors as cs

df = collector.to_polars(lazy=False)

df = df.filter(
    (pl.col("update_count") > 1) & (pl.col("tags").list.contains("rate"))
).select("name", "version", "update_count", "tags")
df

name,version,update_count,tags
str,i64,i64,list[str]
"""evaluate_step""",1,2,"[""time"", ""step""]"
"""boltzmann_selector""",1,2,"[""selector"", ""statistic"", ""time""]"
"""evaluation_count""",1,2,"[""statistic""]"
"""roulette_selector""",1,2,"[""selector"", ""statistic"", ""time""]"
"""evaluate_step""",2,2,"[""time"", ""step""]"
…,…,…,…
"""evaluate_step""",67,2,"[""time"", ""step""]"
"""boltzmann_selector""",67,2,"[""selector"", ""statistic"", ""time""]"
"""gaussian_mutator""",67,2,"[""alterer"", ""mutator"", … ""time""]"
